In [3]:
# -*- coding: utf-8 -*-
"""
Unified 3D adapted SA-PINN benchmark with strict LHS-based error evaluation
and timing.

The implementation follows the adapted SA-PINN structure used in the paper:
1. One network learns the two smooth outer states phi^-(x,y,z), phi^+(x,y,z).
2. One network learns the moving interface h(y,z,t), with h(y,z,0)=0 enforced
   exactly by h=t*N_h(y,z,t).
3. The analytical inner transition corrector is embedded into the trial
   solution.
4. Dirichlet conditions at x=-1 and x=1 are imposed exactly by boundary
   lifting.
5. Periodicity in y and z is imposed softly on the actual boundary faces.
6. Redundant automatic-differentiation work is removed without changing the
   mathematical trial solution or the loss definition.

Benchmark protocol aligned with the 3D DAE/PINN/gPINN codes:
- six fixed seeds;
- fixed training points within each seed;
- the same shared 13,000-point LHS test set selected from the 51^4 reference
  grid by the 3D DAE code;
- 20 warmup evaluations and 200 repeated timed evaluations;
- error computation, CPU transfer, and file I/O excluded from T_eval;
- loss-history transfer and file I/O excluded from T_train;
- LHS predictions saved for every seed;
- full 51^4 prediction saved only for seed 1234, outside all benchmark timing.
"""

import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc
from torch.autograd import grad

# =============================================================================
# 1. Basic settings
# =============================================================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01]

X_MIN, X_MAX = -1.0, 1.0
Y_MIN, Y_MAX = -1.0, 1.0
Z_MIN, Z_MAX = -1.0, 1.0
T_FINAL = 0.5
L_BC, R_BC = -4.0, 2.0
H0_VALUE = 0.0

# Network/training settings from the 3D experiment table.
WIDTH = 10
DEPTH = 6
N_F = 6000
N_PER = 6000
N_I = 6000
EPOCHS = 40000
LEARNING_RATE = 1.0e-3

# Effective point count:
# PDE points + four periodic boundary faces + initial points.
TOTAL_POINTS = N_F + 4 * N_PER + N_I

# Strict LHS evaluation settings.
NUM_SAMPLES = 13000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
REQUIRE_SHARED_LHS_INDEX = True

BASE_PATH = "."
METHOD_NAME = "SAPINN"
SAVE_LHS_PREDICTION = True

# Full 51^4 field output for plotting; excluded from benchmark timings.
SAVE_FULL_FIELD = True
FULL_FIELD_SEED = 1234
FULL_NT = 51
FULL_NX = 51
FULL_NY = 51
FULL_NZ = 51
FULL_FIELD_CHUNK_SIZE = 50000
EXPECTED_REFERENCE_POINTS = FULL_NT * FULL_NX * FULL_NY * FULL_NZ


# =============================================================================
# 2. Utilities
# =============================================================================
def synchronize_backend():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)


def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if arr.size <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))


def compute_error(true_u, pred_u):
    true_vec = np.asarray(true_u, dtype=np.float64).reshape(-1)
    pred_vec = np.asarray(pred_u, dtype=np.float64).reshape(-1)

    if true_vec.shape != pred_vec.shape:
        raise ValueError(
            f"Shape mismatch in error computation: true={true_vec.shape}, "
            f"pred={pred_vec.shape}."
        )

    denom = np.linalg.norm(true_vec)
    if not np.isfinite(denom) or denom <= 0.0:
        raise ValueError("The reference solution has a zero or non-finite L2 norm.")

    diff = pred_vec - true_vec
    e2 = np.linalg.norm(diff) / denom
    einf = np.max(np.abs(diff))
    return float(e2), float(einf)


# =============================================================================
# 3. Physical problem and neural networks
# =============================================================================
def source_f(x, y, z):
    return (
        torch.cos(math.pi * x)
        * torch.cos(math.pi * y)
        * torch.cos(math.pi * z)
    )


def u_init(x, y, z, mu):
    return 3.0 * torch.tanh(x / mu + y + z) - 1.0


class MLP(nn.Module):
    """Fully connected tanh network with `depth` linear layers."""

    def __init__(self, in_dim, out_dim, width, depth):
        super().__init__()
        if depth < 2:
            raise ValueError("depth must be at least 2.")

        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)

        for module in self.net:
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, inputs):
        return self.net(inputs)


class AdaptedSAPINN3D(nn.Module):
    """Adapted 3D semi-analytic PINN.

    The outer network uses raw coordinates (x,y,z), and the interface network
    uses raw coordinates (y,z,t). Periodicity is imposed by the loss.

    The implementation caches the interface jump
        delta = phi_plus(h,y,z) - phi_minus(h,y,z)
    once per forward call. This avoids constructing three identical high-order
    graphs for the base, left-endpoint, and right-endpoint reconstructions.
    """

    def __init__(self, width=WIDTH, depth=DEPTH):
        super().__init__()
        self.N_phi = MLP(3, 2, width, depth)
        self.N_h = MLP(3, 1, width, depth)

    def h(self, y, z, t):
        inputs = torch.cat([y, z, t], dim=1)
        return H0_VALUE + t * self.N_h(inputs)

    def outer(self, x, y, z):
        inputs = torch.cat([x, y, z], dim=1)
        raw = self.N_phi(inputs)

        # Each outer branch satisfies its corresponding non-periodic endpoint
        # value exactly.
        phi_minus = L_BC + (x - X_MIN) * raw[:, 0:1]
        phi_plus = R_BC + (x - X_MAX) * raw[:, 1:2]
        return phi_minus, phi_plus

    def interface_quantities(self, y, z, t, create_graph):
        """Return h and 1-h_y-h_z.

        During training, create_graph=True is necessary because the PDE loss
        differentiates the composite solution twice and then backpropagates to
        the interface-network parameters. During inference it is False.
        """
        h_val = self.h(y, z, t)
        h_y, h_z = grad(
            h_val.sum(),
            (y, z),
            create_graph=create_graph,
            retain_graph=create_graph,
        )
        slope_factor = 1.0 - h_y - h_z
        return h_val, slope_factor

    def interface_jump(self, h_val, y, z):
        phi_minus_h, phi_plus_h = self.outer(h_val, y, z)
        return phi_plus_h - phi_minus_h

    def composite_with_delta(
        self,
        x,
        y,
        z,
        h_val,
        slope_factor,
        delta,
        mu,
    ):
        """Evaluate one left/right composite branch using a cached delta."""
        phi_minus_xyz, phi_plus_xyz = self.outer(x, y, z)

        # Stable logistic representation of the analytical inner corrector.
        arg_left = (h_val - x) * delta * slope_factor / (2.0 * mu)
        arg_right = (x - h_val) * delta * slope_factor / (2.0 * mu)

        q_left = delta * torch.sigmoid(-arg_left)
        q_right = -delta * torch.sigmoid(-arg_right)

        left_branch = phi_minus_xyz + q_left
        right_branch = phi_plus_xyz + q_right
        return torch.where(x <= h_val, left_branch, right_branch)

    def reconstruct_from_interface(
        self,
        x,
        y,
        z,
        h_val,
        slope_factor,
        delta,
        mu,
    ):
        """Construct the boundary-lifted SA-PINN trial solution."""
        base = self.composite_with_delta(
            x, y, z, h_val, slope_factor, delta, mu
        )

        x_left = torch.full_like(x, X_MIN)
        x_right = torch.full_like(x, X_MAX)
        value_left = self.composite_with_delta(
            x_left, y, z, h_val, slope_factor, delta, mu
        )
        value_right = self.composite_with_delta(
            x_right, y, z, h_val, slope_factor, delta, mu
        )

        ell_left = (X_MAX - x) / (X_MAX - X_MIN)
        ell_right = (x - X_MIN) / (X_MAX - X_MIN)

        # Exact Dirichlet boundary lifting in the x direction.
        return (
            base
            + ell_left * (L_BC - value_left)
            + ell_right * (R_BC - value_right)
        )

    def forward(self, x, y, z, t, mu, create_graph=True):
        h_val, slope_factor = self.interface_quantities(
            y, z, t, create_graph=create_graph
        )

        # phi_minus(h), phi_plus(h), and delta are evaluated only once.
        delta = self.interface_jump(h_val, y, z)

        return self.reconstruct_from_interface(
            x, y, z, h_val, slope_factor, delta, mu
        )

    def forward_initial(self, x, y, z, mu):
        """Evaluate the trial solution at t=0 without coordinate AD.

        Since h(y,z,0)=H0_VALUE is imposed exactly and H0_VALUE is constant,
        h_y(y,z,0)=h_z(y,z,0)=0. Therefore the initial-condition loss does not
        need to build the interface-derivative graph.
        """
        h_val = torch.full_like(x, H0_VALUE)
        slope_factor = torch.ones_like(x)
        delta = self.interface_jump(h_val, y, z)

        return self.reconstruct_from_interface(
            x, y, z, h_val, slope_factor, delta, mu
        )


# =============================================================================
# 4. Loss
# =============================================================================
def compute_loss(model, train_data, mu):
    X_f = train_data["X_f"]
    Y_f = train_data["Y_f"]
    Z_f = train_data["Z_f"]
    T_f = train_data["T_f"]

    # Original PDE residual.
    u_f = model(X_f, Y_f, Z_f, T_f, mu, create_graph=True)

    # One autograd call traverses the u_f graph once for all first derivatives.
    u_x, u_y, u_z, u_t = grad(
        u_f.sum(),
        (X_f, Y_f, Z_f, T_f),
        create_graph=True,
    )

    # The diagonal second derivatives must remain separate; combining them in
    # one grad call would also introduce cross derivatives.
    u_xx = grad(u_x.sum(), X_f, create_graph=True)[0]
    u_yy = grad(u_y.sum(), Y_f, create_graph=True)[0]
    u_zz = grad(u_z.sum(), Z_f, create_graph=True)[0]

    # The source is prescribed data. Detach it from the coordinate graph.
    with torch.no_grad():
        f_val = source_f(X_f, Y_f, Z_f)

    residual = (
        mu * (u_xx + u_yy + u_zz)
        - u_t
        + u_f * (u_x + u_y + u_z)
        - f_val
    )
    loss_pde = torch.mean(residual.square())

    # Periodicity on the actual y=-1/y=1 faces. create_graph=True is necessary
    # because the trial solution contains h_y and h_z and the periodic loss must
    # backpropagate through those derivatives to N_h.
    u_y_min = model(
        train_data["X_b"],
        train_data["Y_b_min"],
        train_data["Z_b"],
        train_data["T_b"],
        mu,
        create_graph=True,
    )
    u_y_max = model(
        train_data["X_b"],
        train_data["Y_b_max"],
        train_data["Z_b"],
        train_data["T_b"],
        mu,
        create_graph=True,
    )
    loss_per_y = torch.mean((u_y_min - u_y_max).square())

    # Periodicity on the actual z=-1/z=1 faces.
    u_z_min = model(
        train_data["X_b"],
        train_data["Y_b"],
        train_data["Z_b_min"],
        train_data["T_b"],
        mu,
        create_graph=True,
    )
    u_z_max = model(
        train_data["X_b"],
        train_data["Y_b"],
        train_data["Z_b_max"],
        train_data["T_b"],
        mu,
        create_graph=True,
    )
    loss_per_z = torch.mean((u_z_min - u_z_max).square())

    # At t=0, h=H0_VALUE and h_y=h_z=0 exactly, so no interface-coordinate
    # derivative graph is needed for the initial-condition loss.
    u_i = model.forward_initial(
        train_data["X_i"],
        train_data["Y_i"],
        train_data["Z_i"],
        mu,
    )
    loss_ic = torch.mean((u_i - train_data["U_i_target"]).square())

    total_loss = loss_pde + loss_per_y + loss_per_z + loss_ic
    return total_loss


# =============================================================================
# 5. Training-data construction
# =============================================================================
def uniform_column(n, lower, upper, requires_grad=False):
    tensor = (
        torch.rand(n, 1, device=DEVICE, dtype=DTYPE) * (upper - lower) + lower
    )
    return tensor.requires_grad_(requires_grad)


def constant_column(n, value, requires_grad=False):
    tensor = torch.full(
        (n, 1), value, device=DEVICE, dtype=DTYPE
    )
    return tensor.requires_grad_(requires_grad)


def build_training_data(mu):
    # Interior residual points.
    X_f = uniform_column(N_F, X_MIN, X_MAX, requires_grad=True)
    Y_f = uniform_column(N_F, Y_MIN, Y_MAX, requires_grad=True)
    Z_f = uniform_column(N_F, Z_MIN, Z_MAX, requires_grad=True)
    T_f = uniform_column(N_F, 0.0, T_FINAL, requires_grad=True)

    # Periodic boundary parameterization. Y_b and Z_b require gradients because
    # the trial solution contains h_y and h_z even on boundary evaluations.
    X_b = uniform_column(N_PER, X_MIN, X_MAX, requires_grad=False)
    Y_b = uniform_column(N_PER, Y_MIN, Y_MAX, requires_grad=True)
    Z_b = uniform_column(N_PER, Z_MIN, Z_MAX, requires_grad=True)
    T_b = uniform_column(N_PER, 0.0, T_FINAL, requires_grad=False)

    Y_b_min = constant_column(N_PER, Y_MIN, requires_grad=True)
    Y_b_max = constant_column(N_PER, Y_MAX, requires_grad=True)
    Z_b_min = constant_column(N_PER, Z_MIN, requires_grad=True)
    Z_b_max = constant_column(N_PER, Z_MAX, requires_grad=True)

    # Initial points. Because h(y,z,0)=H0_VALUE and h_y=h_z=0 exactly,
    # no coordinate gradients are needed on the initial surface.
    X_i = uniform_column(N_I, X_MIN, X_MAX, requires_grad=False)
    Y_i = uniform_column(N_I, Y_MIN, Y_MAX, requires_grad=False)
    Z_i = uniform_column(N_I, Z_MIN, Z_MAX, requires_grad=False)
    T_i = constant_column(N_I, 0.0, requires_grad=False)
    U_i_target = u_init(X_i, Y_i, Z_i, mu).detach()

    return {
        "X_f": X_f,
        "Y_f": Y_f,
        "Z_f": Z_f,
        "T_f": T_f,
        "X_b": X_b,
        "Y_b": Y_b,
        "Z_b": Z_b,
        "T_b": T_b,
        "Y_b_min": Y_b_min,
        "Y_b_max": Y_b_max,
        "Z_b_min": Z_b_min,
        "Z_b_max": Z_b_max,
        "X_i": X_i,
        "Y_i": Y_i,
        "Z_i": Z_i,
        "T_i": T_i,
        "U_i_target": U_i_target,
    }


# =============================================================================
# 6. Inference helper
# =============================================================================
def freeze_model_parameters(model):
    """Disable parameter-gradient tracking after training."""
    for parameter in model.parameters():
        parameter.requires_grad_(False)


def eval_u(model, x_eval, y_eval, z_eval, t_eval, mu):
    """Evaluate SA-PINN with only the coordinate AD required for h_y,h_z.

    Model parameters must be frozen before timed evaluation. Only the interface
    network is differentiated with respect to y and z. Once h, h_y, and h_z are
    obtained, the outer network and the full reconstruction are evaluated under
    torch.no_grad(), so the returned prediction has no grad_fn.
    """
    x_val = x_eval.detach()
    y_val = y_eval.detach()
    z_val = z_eval.detach()
    t_val = t_eval.detach()

    with torch.enable_grad():
        # detach() creates fresh leaf views without copying the coordinate data.
        y_ad = y_val.detach().requires_grad_(True)
        z_ad = z_val.detach().requires_grad_(True)

        h_val = model.h(y_ad, z_ad, t_val)
        h_y, h_z = grad(
            h_val.sum(),
            (y_ad, z_ad),
            create_graph=False,
            retain_graph=False,
        )
        slope_factor = 1.0 - h_y - h_z

    with torch.no_grad():
        h_val = h_val.detach()
        slope_factor = slope_factor.detach()
        delta = model.interface_jump(h_val, y_val, z_val)
        prediction = model.reconstruct_from_interface(
            x_val,
            y_val,
            z_val,
            h_val,
            slope_factor,
            delta,
            mu,
        )

    return prediction


# =============================================================================
# 7. LHS reference-data utilities
# =============================================================================
def get_target_col(df):
    if "u" in df.columns:
        return "u"
    if "u0" in df.columns:
        return "u0"
    raise ValueError(
        "Cannot identify the reference-solution column. "
        f"Available columns: {list(df.columns)}"
    )


def load_true_solution(mu):
    mu_id = int(round(-math.log10(mu)))
    candidates = [
        f"3d_U0_true_mu{mu:.0e}.csv",
        f"3d_U0_all_t_u_x_y_z_t_mu{mu_id}_51_mathematica_619.csv",
    ]
    if np.isclose(mu, 1.0e-2):
        candidates.append("3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv")

    for filename in candidates:
        path = os.path.join(BASE_PATH, filename)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df.columns = [str(col).lower().strip() for col in df.columns]

            required = {"t", "x", "y", "z"}
            missing = required.difference(df.columns)
            if missing:
                raise ValueError(
                    f"Reference file {filename} is missing columns: "
                    f"{sorted(missing)}"
                )

            df = df.sort_values(by=["t", "x", "y", "z"]).reset_index(drop=True)
            return df, filename

    raise FileNotFoundError(
        "Cannot find the 3D reference solution. Tried: " + ", ".join(candidates)
    )


def build_or_load_lhs_test_set(mu):
    df_true, reference_filename = load_true_solution(mu)

    if len(df_true) != EXPECTED_REFERENCE_POINTS:
        raise ValueError(
            f"Reference file {reference_filename} contains {len(df_true)} rows, "
            f"but a 51^4 grid must contain {EXPECTED_REFERENCE_POINTS} rows."
        )

    index_file = f"3d_LHS_sample_indices_mu{mu:.0e}.npy"
    sample_indices = None

    if os.path.exists(index_file):
        loaded = np.asarray(np.load(index_file), dtype=np.int64).reshape(-1)
        valid = (
            loaded.size == NUM_SAMPLES
            and np.unique(loaded).size == NUM_SAMPLES
            and loaded.min() >= 0
            and loaded.max() < len(df_true)
        )
        if valid:
            sample_indices = loaded
            print(f"[mu={mu}] Loaded valid shared LHS indices from {index_file}.")
        else:
            print(f"[mu={mu}] Existing LHS index file is invalid.")

    if sample_indices is None and REQUIRE_SHARED_LHS_INDEX:
        raise FileNotFoundError(
            f"The shared DAE/PINN/gPINN/SA-PINN LHS index file {index_file} "
            "is missing or invalid. Run the 3D DAE code once to create it, "
            "or set REQUIRE_SHARED_LHS_INDEX=False to generate it here."
        )

    if sample_indices is None:
        coordinate_columns = ["t", "x", "y", "z"]
        all_points = df_true[coordinate_columns].to_numpy(dtype=np.float64)
        lower = all_points.min(axis=0)
        upper = all_points.max(axis=0)
        kdtree = cKDTree(all_points)

        selected = []
        used = set()
        batch_id = 0

        while len(selected) < NUM_SAMPLES and batch_id < 100:
            sampler = qmc.LatinHypercube(d=4, seed=LHS_SEED + batch_id)
            lhs_unit = sampler.random(n=NUM_SAMPLES)
            lhs_scaled = qmc.scale(lhs_unit, lower, upper)
            _, candidate_indices = kdtree.query(lhs_scaled, k=1)

            for idx in np.asarray(candidate_indices).reshape(-1):
                idx = int(idx)
                if idx not in used:
                    used.add(idx)
                    selected.append(idx)
                    if len(selected) == NUM_SAMPLES:
                        break
            batch_id += 1

        if len(selected) < NUM_SAMPLES:
            remaining = np.setdiff1d(
                np.arange(len(df_true), dtype=np.int64),
                np.asarray(selected, dtype=np.int64),
                assume_unique=False,
            )
            rng = np.random.default_rng(LHS_SEED)
            fill = rng.choice(
                remaining,
                size=NUM_SAMPLES - len(selected),
                replace=False,
            )
            selected.extend(int(idx) for idx in fill)

        sample_indices = np.asarray(selected, dtype=np.int64)
        np.save(index_file, sample_indices)
        print(
            f"[mu={mu}] Generated and saved {NUM_SAMPLES} unique LHS indices "
            f"to {index_file}."
        )

    sampled = df_true.iloc[sample_indices]
    t_np = sampled["t"].to_numpy(dtype=np.float64).reshape(-1, 1)
    x_np = sampled["x"].to_numpy(dtype=np.float64).reshape(-1, 1)
    y_np = sampled["y"].to_numpy(dtype=np.float64).reshape(-1, 1)
    z_np = sampled["z"].to_numpy(dtype=np.float64).reshape(-1, 1)

    true_col = get_target_col(df_true)
    true_np = sampled[true_col].to_numpy(dtype=np.float64).reshape(-1)

    print(
        f"[mu={mu}] Reference={reference_filename} | "
        f"reference rows={len(df_true)}=51^4 | "
        f"shared index file={index_file} | LHS points={sample_indices.size}"
    )

    return {
        "x_lhs": torch.tensor(x_np, dtype=DTYPE, device=DEVICE),
        "y_lhs": torch.tensor(y_np, dtype=DTYPE, device=DEVICE),
        "z_lhs": torch.tensor(z_np, dtype=DTYPE, device=DEVICE),
        "t_lhs": torch.tensor(t_np, dtype=DTYPE, device=DEVICE),
        "true_lhs_np": true_np,
        "t_np": t_np,
        "x_np": x_np,
        "y_np": y_np,
        "z_np": z_np,
        "n_test": int(sample_indices.size),
        "reference_file": reference_filename,
        "reference_rows": int(len(df_true)),
        "lhs_index_file": index_file,
    }


# =============================================================================
# 8. Full-field reconstruction for seed 1234
# =============================================================================
def save_full_field_prediction(model, mu, seed):
    """Save the full 51^4 field in the original meshgrid-C flattening order.

    Row order matches
        T, X, Y, Z = np.meshgrid(t, x, y, z, indexing="ij")
        flatten(order="C")
    so t is the slowest index and z is the fastest index.

    This function is called outside T_train, T_eval, and T_total.
    """
    if not SAVE_FULL_FIELD or seed != FULL_FIELD_SEED:
        return None

    total = FULL_NT * FULL_NX * FULL_NY * FULL_NZ
    t_vals = np.linspace(0.0, T_FINAL, FULL_NT, dtype=np.float64)
    x_vals = np.linspace(X_MIN, X_MAX, FULL_NX, dtype=np.float64)
    y_vals = np.linspace(Y_MIN, Y_MAX, FULL_NY, dtype=np.float64)
    z_vals = np.linspace(Z_MIN, Z_MAX, FULL_NZ, dtype=np.float64)

    filename = f"3d_{METHOD_NAME}_mu{mu:.2f}_U0_predicted_seed{seed}.csv"
    first_chunk = True

    print(
        f"  > Saving full 51^4 SA-PINN prediction for seed={seed}: "
        f"{total} rows -> {filename}"
    )

    model.eval()
    for start in range(0, total, FULL_FIELD_CHUNK_SIZE):
        stop = min(start + FULL_FIELD_CHUNK_SIZE, total)
        flat = np.arange(start, stop, dtype=np.int64)

        iz = flat % FULL_NZ
        q = flat // FULL_NZ
        iy = q % FULL_NY
        q //= FULL_NY
        ix = q % FULL_NX
        it = q // FULL_NX

        t_chunk_np = t_vals[it]
        x_chunk_np = x_vals[ix]
        y_chunk_np = y_vals[iy]
        z_chunk_np = z_vals[iz]

        x_chunk = torch.tensor(
            x_chunk_np.reshape(-1, 1), dtype=DTYPE, device=DEVICE
        )
        y_chunk = torch.tensor(
            y_chunk_np.reshape(-1, 1), dtype=DTYPE, device=DEVICE
        )
        z_chunk = torch.tensor(
            z_chunk_np.reshape(-1, 1), dtype=DTYPE, device=DEVICE
        )
        t_chunk = torch.tensor(
            t_chunk_np.reshape(-1, 1), dtype=DTYPE, device=DEVICE
        )

        u_chunk = (
            eval_u(model, x_chunk, y_chunk, z_chunk, t_chunk, mu)
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        df_chunk = pd.DataFrame(
            {
                "u": u_chunk,
                "x": x_chunk_np,
                "y": y_chunk_np,
                "z": z_chunk_np,
                "t": t_chunk_np,
            }
        )
        df_chunk.to_csv(
            filename,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
        )
        first_chunk = False

    print(f"  > Full-field prediction saved: {filename}")
    return filename


# =============================================================================
# 9. Main benchmark
# =============================================================================
def main():
    print("\n" + "=" * 92)
    print("Starting 3D adapted SA-PINN benchmark with strict LHS timing and error")
    print(
        f"Device: {DEVICE} | dtype: {DTYPE} | "
        f"LHS test points: {NUM_SAMPLES} | LHS seed: {LHS_SEED}"
    )
    print(
        f"Warmup: {EVAL_WARMUP} | repeats: {EVAL_REPEAT} | "
        f"epochs: {EPOCHS} | depth: {DEPTH} | width: {WIDTH}"
    )
    print(
        f"Training sizes: N_f={N_F}, N_per={N_PER}, N_i={N_I} | "
        f"effective points={TOTAL_POINTS}"
    )
    print("=" * 92 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for mu in MU_LIST:
        print("\n" + "=" * 92)
        print(f"Starting 3D {METHOD_NAME} for mu={mu}")
        print("=" * 92)

        data_mu = lhs_data[mu]
        x_lhs = data_mu["x_lhs"]
        y_lhs = data_mu["y_lhs"]
        z_lhs = data_mu["z_lhs"]
        t_lhs = data_mu["t_lhs"]
        true_lhs_np = data_mu["true_lhs_np"]
        n_test = data_mu["n_test"]

        for seed in SEEDS:
            print(f"\n--- Running Seed: {seed} ---")
            set_seed(seed)

            train_data = build_training_data(mu)
            total_point_steps = EPOCHS * TOTAL_POINTS

            print(
                "  [Points] "
                f"PDE={N_F} | periodic faces=4x{N_PER}={4 * N_PER} | "
                f"IC={N_I} | total={TOTAL_POINTS}"
            )

            model = AdaptedSAPINN3D().to(device=DEVICE, dtype=DTYPE)
            optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
            model_parameters = tuple(model.parameters())
            loss_history_gpu = []

            # -------------------------------------------------------------
            # Phase A: training. No CPU transfer or file I/O is timed.
            # -------------------------------------------------------------
            model.train()
            synchronize_backend()
            train_start = time.perf_counter()

            for _ in range(EPOCHS):
                optimizer.zero_grad(set_to_none=True)
                loss = compute_loss(model, train_data, mu)
                # Restrict gradient accumulation to trainable parameters.
                # The fixed collocation tensors are leaf variables only because
                # automatic differentiation with respect to coordinates is needed;
                # their .grad fields must not accumulate across epochs.
                loss.backward(inputs=model_parameters)
                optimizer.step()
                loss_history_gpu.append(loss.detach())

            synchronize_backend()
            T_train = time.perf_counter() - train_start

            loss_history = (
                torch.stack(loss_history_gpu).cpu().numpy().astype(np.float64)
            )
            e_loss = float(loss_history[-1])

            loss_filename = (
                f"3d_{METHOD_NAME}_loss_history_mu{mu:.0e}_seed{seed}.npy"
            )
            np.save(loss_filename, loss_history)

            T_train_per_iter_ms = T_train * 1.0e3 / EPOCHS
            T_train_per_iter_point_us = (
                T_train * 1.0e6 / total_point_steps
            )

            print(
                f"  > Trained: T_train={T_train:.2f}s | "
                f"e_loss={e_loss:.3e} | loss_file={loss_filename}"
            )

            # -------------------------------------------------------------
            # Phase B: strict LHS evaluation timing.
            # SA-PINN evaluation necessarily includes h_y and h_z AD.
            # -------------------------------------------------------------
            model.eval()
            freeze_model_parameters(model)

            for _ in range(EVAL_WARMUP):
                warmup_output = eval_u(
                    model, x_lhs, y_lhs, z_lhs, t_lhs, mu
                )
                del warmup_output

            synchronize_backend()
            eval_start = time.perf_counter()

            for _ in range(EVAL_REPEAT):
                timed_output = eval_u(
                    model, x_lhs, y_lhs, z_lhs, t_lhs, mu
                )
                del timed_output

            synchronize_backend()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            # -------------------------------------------------------------
            # Phase C: one extra prediction for errors, outside T_eval.
            # -------------------------------------------------------------
            u_pred_lhs = (
                eval_u(model, x_lhs, y_lhs, z_lhs, t_lhs, mu)
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            e2, einf = compute_error(true_lhs_np, u_pred_lhs)
            T_total = T_train + T_eval

            print(
                f"    -> [mu={mu}] N_test={n_test} | "
                f"T_eval={T_eval:.6e}s | e2={e2:.3e} | einf={einf:.3e}"
            )

            if SAVE_LHS_PREDICTION:
                lhs_filename = (
                    f"3d_{METHOD_NAME}_U0_predicted_LHS_"
                    f"mu{mu:.0e}_seed{seed}.csv"
                )
                pd.DataFrame(
                    {
                        "t": data_mu["t_np"].reshape(-1),
                        "x": data_mu["x_np"].reshape(-1),
                        "y": data_mu["y_np"].reshape(-1),
                        "z": data_mu["z_np"].reshape(-1),
                        "u": u_pred_lhs,
                    }
                ).to_csv(lhs_filename, index=False)
            else:
                lhs_filename = ""

            metrics[mu].append(
                {
                    "Seed": seed,
                    "N_test": n_test,
                    "e_loss": e_loss,
                    "e2": e2,
                    "einf": einf,
                    "T_train": T_train,
                    "T_eval": T_eval,
                    "T_total": T_total,
                    "T_train_per_iter_ms": T_train_per_iter_ms,
                    "T_train_per_iter_point_us": T_train_per_iter_point_us,
                    "total_trained_steps": EPOCHS,
                    "total_point_steps": total_point_steps,
                    "final_residual_points": TOTAL_POINTS,
                    "eval_warmup": EVAL_WARMUP,
                    "eval_repeat": EVAL_REPEAT,
                    "lhs_prediction_file": lhs_filename,
                }
            )

            # Full field for plotting only; excluded from all timings.
            save_full_field_prediction(model, mu, seed)

            # Release large graphs/tensors before the next seed.
            del model, optimizer, model_parameters, train_data, loss_history_gpu
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    # =====================================================================
    # Summary
    # =====================================================================
    print("\n" + "=" * 92)
    print("ALL SEEDS COMPLETED. GENERATING SUMMARY TABLES.")
    print("=" * 92 + "\n")

    for mu in MU_LIST:
        df_mu = pd.DataFrame(metrics[mu])
        summary_filename = (
            f"3d_{METHOD_NAME}_mu{mu:.0e}_Metrics_Summary.csv"
        )
        df_mu.to_csv(summary_filename, index=False)

        cols = [
            "e_loss",
            "e2",
            "einf",
            "T_train",
            "T_eval",
            "T_total",
            "T_train_per_iter_ms",
            "T_train_per_iter_point_us",
            "total_trained_steps",
            "total_point_steps",
            "final_residual_points",
            "N_test",
        ]
        stats = {col: mean_std(df_mu[col].values) for col in cols}

        print(
            f"### Results for 3D {METHOD_NAME}, mu={mu} "
            "[Mean +/- Sample Std] ###"
        )
        print(f"N_test: {stats['N_test'][0]:.0f} +/- {stats['N_test'][1]:.0f}")
        print(
            f"e_loss: {stats['e_loss'][0]:.3e} +/- "
            f"{stats['e_loss'][1]:.3e}"
        )
        print(f"e_2: {stats['e2'][0]:.3e} +/- {stats['e2'][1]:.3e}")
        print(
            f"e_inf: {stats['einf'][0]:.3e} +/- "
            f"{stats['einf'][1]:.3e}"
        )
        print(
            f"T_train (s): {stats['T_train'][0]:.2f} +/- "
            f"{stats['T_train'][1]:.2f}"
        )
        print(
            f"T_eval (s): {stats['T_eval'][0]:.6e} +/- "
            f"{stats['T_eval'][1]:.6e}"
        )
        print(
            f"T_total (s): {stats['T_total'][0]:.2f} +/- "
            f"{stats['T_total'][1]:.2f}"
        )
        print(
            f"T_train/iter (ms): "
            f"{stats['T_train_per_iter_ms'][0]:.4f} +/- "
            f"{stats['T_train_per_iter_ms'][1]:.4f}"
        )
        print(
            f"T_train/(iter*pt) (us): "
            f"{stats['T_train_per_iter_point_us'][0]:.4f} +/- "
            f"{stats['T_train_per_iter_point_us'][1]:.4f}"
        )
        print(
            f"Optimization steps: {stats['total_trained_steps'][0]:.0f} +/- "
            f"{stats['total_trained_steps'][1]:.0f}"
        )
        print(
            f"Point-iterations: {stats['total_point_steps'][0]:.0f} +/- "
            f"{stats['total_point_steps'][1]:.0f}"
        )
        print(
            f"Final effective points: "
            f"{stats['final_residual_points'][0]:.0f} +/- "
            f"{stats['final_residual_points'][1]:.0f}"
        )
        print(f"Summary file: {summary_filename}\n")


if __name__ == "__main__":
    main()



Starting 3D adapted SA-PINN benchmark with strict LHS timing and error
Device: cuda:0 | dtype: torch.float32 | LHS test points: 13000 | LHS seed: 1234
Warmup: 20 | repeats: 200 | epochs: 40000 | depth: 6 | width: 10
Training sizes: N_f=6000, N_per=6000, N_i=6000 | effective points=36000

[mu=0.01] Loaded valid shared LHS indices from 3d_LHS_sample_indices_mu1e-02.npy.
[mu=0.01] Reference=3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv | reference rows=6765201=51^4 | shared index file=3d_LHS_sample_indices_mu1e-02.npy | LHS points=13000

Starting 3D SAPINN for mu=0.01

--- Running Seed: 33 ---
  [Points] PDE=6000 | periodic faces=4x6000=24000 | IC=6000 | total=36000
  > Trained: T_train=6828.39s | e_loss=6.845e-02 | loss_file=3d_SAPINN_loss_history_mu1e-02_seed33.npy
    -> [mu=0.01] N_test=13000 | T_eval=3.743938e-03s | e2=4.484e-02 | einf=4.467e+00

--- Running Seed: 99 ---
  [Points] PDE=6000 | periodic faces=4x6000=24000 | IC=6000 | total=36000
  > Trained: T_train=7113.84s | e_lo

In [2]:
pip install scipy -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/8e/6d/41991e503e51fc1134502694c5fa7a1671501a17ffa12716a4a9151af3df/scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Note: you may need to restart the kernel to use updated packages.
